In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
%pip install -U dagshub mlflow skops --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 96.5 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 44.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
import dagshub
dagshub.init(repo_owner='gvakh23', repo_name='ML_assignment2', mlflow=True)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=53440ed5-13bc-401a-94ba-98ed1097055f&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=e270a17872136bdff1dd4489322ba25f236520b96bb91bc0f1ede4d164ab7cd1




Output()

Accessing as gvakh23

Initialized MLflow to track repo "gvakh23/ML_assignment2"

Repository gvakh23/ML_assignment2 initialized!

In [4]:
import mlflow
print(mlflow.get_tracking_uri())

https://dagshub.com/gvakh23/ML_assignment2.mlflow


In [5]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

train_trans  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_ident  = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

train = train_trans.merge(train_ident, on='TransactionID', how='left')


In [6]:
train_sorted = train.sort_values('TransactionDT')

val_size = int(len(train_sorted) * 0.2)
train_df = train_sorted.iloc[:-val_size].copy()
val_df   = train_sorted.iloc[-val_size:].copy()

print(f"Train split : {train_df.shape}, fraud rate: {train_df['isFraud'].mean():.2%}")
print(f"Val   split : {val_df.shape},   fraud rate: {val_df['isFraud'].mean():.2%}")


Train split : (472432, 434), fraud rate: 3.51%
Val   split : (118108, 434),   fraud rate: 3.44%


## Cleaning

In [64]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold
class Cleaner(BaseEstimator, TransformerMixin):
    """
    Drops high-null columns, encodes M columns (T/F→1/0),
    fills remaining NaN with -999.
    All thresholds computed on train only.
    """
 
    def __init__(self, null_threshold=0.9, fill_value=-999):
        self.null_threshold = null_threshold
        self.fill_value     = fill_value
 
    def fit(self, X, y=None):
        print("clean_in")

        df = X.copy()
 
        null_rates = df.isnull().mean()
        self.cols_to_drop_ = null_rates[null_rates > self.null_threshold].index.tolist()
        self.cols_to_drop_ = [c for c in self.cols_to_drop_
                               if c not in ['isFraud', 'TransactionID']]
 
        # Learn M column names
        self.m_cols_ = [c for c in df.columns if c.startswith('M')]
 
        with mlflow.start_run(run_name='DecisionTree_Cleaning', nested=True):
            mlflow.log_param('null_threshold',  self.null_threshold)
            mlflow.log_param('fill_value',      self.fill_value)
            mlflow.log_param('fill_reason',     'tree_model_out_of_range_sentinel')
            mlflow.log_param('m_col_encoding',  'T=1_F=0_NaN=fill_value')
            mlflow.log_metric('cols_dropped',   len(self.cols_to_drop_))
            mlflow.log_metric('cols_remaining', df.shape[1] - len(self.cols_to_drop_))
            mlflow.log_metric('rows',           df.shape[0])
            mlflow.log_metric('fraud_rate',     float(y.mean()) if y is not None else -1)
        print(f"[Cleaner] Done — dropped {len(self.cols_to_drop_)} cols, {df.shape[0]} rows")
        return self
 
    def transform(self, X):
        print("clean trans in")
        df = X.copy()
 
        # Drop high-null cols
        df.drop(columns=[c for c in self.cols_to_drop_ if c in df.columns],
                inplace=True, errors='ignore')
 
        # M columns T/F → 1/0
        for col in self.m_cols_:
            if col in df.columns:
                df[col] = df[col].map({'T': 1, 'F': 0})
 
        # Fill NaN
        df.fillna(self.fill_value, inplace=True)
 
        return df

## Feature Engineering

In [65]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.d_cols      = ['D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D15']
        self.freq_cols   = ['card1', 'card2', 'addr1', 'uid', 'uid2', 'P_emaildomain']
        self.agg_cols    = ['card1', 'uid']          # removed card2/addr1 — less useful
        self.agg_targets = ['TransactionAmt', 'D1', 'C1', 'C2', 'C13']

    def fit(self, X, y=None):
        df = X.copy()

        # 1. D normalization mins
        self.d_min_map_ = {}
        for d in self.d_cols:
            if d in df.columns:
                self.d_min_map_[d] = df.groupby('card1')[d].min()

        # 2. Build UIDs
        df['uid']  = df['card1'].astype(str) + '_' + df['card2'].astype(str)
        # uid2: use D1 only if it's not -999 (missing)
        d1_str     = df['D1'].apply(lambda x: str(int(x)) if x != -999 else 'NA')
        df['uid2'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str) + '_' + d1_str

        # 3. Frequency maps
        self.freq_maps_ = {}
        for col in self.freq_cols:
            if col in df.columns:
                self.freq_maps_[col] = df[col].value_counts()

        # 4. Aggregation maps — store as dict of Series for fast map()
        self.agg_maps_ = {}
        for col in self.agg_cols:
            if col not in df.columns:
                continue
            self.agg_maps_[col] = {}
            for target in self.agg_targets:
                if target not in df.columns:
                    continue
                self.agg_maps_[col][target] = {}
                grp = df.groupby(col)[target]
                self.agg_maps_[col][target]['mean'] = grp.mean()
                self.agg_maps_[col][target]['std']  = grp.std()
                self.agg_maps_[col][target]['max']  = grp.max()

        # 5. UID label encoders — store as plain dict for fastest mapping
        self.uid_maps_ = {}
        for col in ['uid', 'uid2']:
            if col in df.columns:
                unique_vals = df[col].astype(str).unique()
                self.uid_maps_[col] = {v: i for i, v in enumerate(unique_vals)}

        # 6. Email suffix encoder
        if 'P_emaildomain' in df.columns:
            suffixes = df['P_emaildomain'].astype(str).str.split('.').str[-1]
            unique_sfx = suffixes.unique()
            self.email_suffix_map_ = {v: i for i, v in enumerate(unique_sfx)}
        else:
            self.email_suffix_map_ = {}


        with mlflow.start_run(run_name='DecisionTree_Engineering', nested=True):
            mlflow.log_param('time_features',      'hour, day')
            mlflow.log_param('log_transform',      'TransactionAmt_log, TransactionAmt_decimal')
            mlflow.log_param('d_cols_normalized',  str([d for d in self.d_cols if d in self.d_min_map_]))
            mlflow.log_param('d_norm_method',      'subtract_card1_group_min_train_only')
            mlflow.log_param('uid',                'card1+card2')
            mlflow.log_param('uid2',               'card1+addr1+D1 (NA if D1 missing)')
            mlflow.log_param('freq_cols',          str(self.freq_cols))
            mlflow.log_param('agg_cols',           str(self.agg_cols))
            mlflow.log_param('agg_targets',        str(self.agg_targets))
            mlflow.log_param('email_features',     'same_email, P_email_suffix')
            mlflow.log_metric('d_cols_fitted',     len(self.d_min_map_))
            mlflow.log_metric('freq_cols_fitted',  len(self.freq_maps_))
            mlflow.log_metric('uid_maps_fitted',   len(self.uid_maps_))

        return self

    def transform(self, X):
        df = X.copy()

        # ── Time features ─────────────────────────────────────
        df['hour']  = (df['TransactionDT'] / 3600) % 24
        df['day']   = (df['TransactionDT'] / (3600 * 24)) % 7

        # ── Log transform ─────────────────────────────────────
        df['TransactionAmt_log']     = np.log1p(df['TransactionAmt'])
        df['TransactionAmt_decimal'] = df['TransactionAmt'] - df['TransactionAmt'].astype(int)

        # ── Email features ────────────────────────────────────
        if 'P_emaildomain' in df.columns and 'R_emaildomain' in df.columns:
            df['same_email'] = (df['P_emaildomain'] == df['R_emaildomain']).astype(int)
            # Encode suffix using saved map (fast dict lookup, no lambda)
            suffixes = df['P_emaildomain'].astype(str).str.split('.').str[-1]
            df['P_email_suffix'] = suffixes.map(self.email_suffix_map_).fillna(-1).astype(int)

        # ── D normalization ───────────────────────────────────
        for d, min_map in self.d_min_map_.items():
            if d in df.columns:
                df[f'{d}_norm'] = df[d] - df['card1'].map(min_map).fillna(0)

        # ── UIDs ──────────────────────────────────────────────
        df['uid']  = df['card1'].astype(str) + '_' + df['card2'].astype(str)
        d1_str     = df['D1'].apply(lambda x: str(int(x)) if x != -999 else 'NA')
        df['uid2'] = df['card1'].astype(str) + '_' + df['addr1'].astype(str) + '_' + d1_str

        # ── Frequency encoding ────────────────────────────────
        for col, freq_map in self.freq_maps_.items():
            if col in df.columns:
                df[f'{col}_freq'] = df[col].map(freq_map).fillna(0)

        # ── Aggregations ──────────
        for col, targets in self.agg_maps_.items():
            if col not in df.columns:
                continue
            for target, stats in targets.items():
                for stat, series in stats.items():
                    df[f'{col}_{target}_{stat}'] = df[col].map(series).fillna(-999)

        # ── UID encoding ──────────────
        for col, uid_map in self.uid_maps_.items():
            if col in df.columns:
                df[col] = df[col].astype(str).map(uid_map).fillna(-1).astype(int)

        return df




## Categorical Encoding

In [66]:
import pandas as pd
import mlflow
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import LabelEncoder

class CategoricalEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, ohe_threshold=10):
        self.ohe_threshold = ohe_threshold
        self.device_map_ = {'desktop': 1, 'mobile': 0}
        self.ohe_info_ = {}
        self.le_encoders_ = {}
        self.le_cols_ = []

    def fit(self, X, y=None):
        print("encode_in")

        df = X.copy()
        
        # 1. Identify Categorical Columns (excluding IDs/Targets)
        all_cat_cols = [c for c in df.select_dtypes(include='object').columns 
                        if c not in ['TransactionID', 'isFraud', 'DeviceType']]

        # 2. Split logic: OHE vs Label Encode based on threshold
        for col in all_cat_cols:
            n_unique = df[col].nunique()
            if n_unique < self.ohe_threshold:
                self.ohe_info_[col] = df[col].unique().tolist()
            else:
                le = LabelEncoder()
                le.fit(df[col].astype(str))
                self.le_encoders_[col] = le
                self.le_cols_.append(col)

        # MLflow logging (matches your previous class style)
        with mlflow.start_run(run_name='DecisionTree_Categorical_Encoding', nested=True):
            mlflow.log_param('ohe_threshold', self.ohe_threshold)
            mlflow.log_param('ohe_cols', list(self.ohe_info_.keys()))
            mlflow.log_param('le_cols', self.le_cols_)
            mlflow.log_metric('ohe_count', len(self.ohe_info_))
            mlflow.log_metric('le_count', len(self.le_cols_))
        print(f"[CategoricalEncoder] Done — OHE cols: {len(self.ohe_info_)}, LE cols: {len(self.le_cols_)}")

        return self

    def transform(self, X):
        print("encode trans in")

        df = X.copy()

        # ── BINARY: DeviceType ──
        if 'DeviceType' in df.columns:
            df['DeviceType'] = df['DeviceType'].map(self.device_map_).fillna(-999)

        # ── ONE-HOT ENCODING ──
        for col, categories in self.ohe_info_.items():
            if col in df.columns:
        
                dummies = pd.get_dummies(df[col], prefix=col)
        
                expected_cols = [f"{col}_{cat}" for cat in categories]
                dummies = dummies.reindex(columns=expected_cols, fill_value=0)
        
                df = pd.concat([df, dummies], axis=1)
                df.drop(columns=[col], inplace=True)
                
        # ── LABEL ENCODING ──
        for col, le in self.le_encoders_.items():
            if col in df.columns:
                # Handle unseen categories by mapping to -1
                known_classes = set(le.classes_)
                mapping = {cls: idx for idx, cls in enumerate(le.classes_)}
                df[col] = df[col].astype(str).map(mapping).fillna(-1)

        return df

## Feature Selection

In [67]:
import pandas as pd
import numpy as np
import mlflow
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import VarianceThreshold

class FeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, variance_threshold=0.01):
        self.variance_threshold = variance_threshold
        self.drop_cols_ = []
        self.feature_cols_ = []

    def fit(self, X, y=None):
        print("select_in")

        df = X.copy()
        numeric_df = df.select_dtypes(include=[np.number])
        
        # ──  Variance Threshold ──
        vt = VarianceThreshold(threshold=self.variance_threshold)
        vt.fit(numeric_df)
        low_var_cols = numeric_df.columns[~vt.get_support()].tolist()

        # ──  V-Column NaN-Group Selection ──
        # In this dataset, V columns are often redundant. 
        # Group them by their 'NaN pattern' and keep only the one strongest with target.
        v_cols = [c for c in df.columns if c.startswith('V')]
        
        null_patterns = {}
        for col in v_cols:
            pattern = int((df[col] == -999).sum())
            null_patterns.setdefault(pattern, []).append(col)

        best_per_group = []
        if y is not None:
            # Convert y to series for easy correlation mapping
            y_s = pd.Series(y.values if hasattr(y, 'values') else np.array(y), index=df.index)
            for pattern, cols in null_patterns.items():
                corrs = df[cols].corrwith(y_s).abs().fillna(0)
                best_per_group.append(corrs.idxmax())
        else:
            for pattern, cols in null_patterns.items():
                best_per_group.append(cols[0])

        v_to_drop = [c for c in v_cols if c not in best_per_group]
        
        # Combine all columns to remove
        self.drop_cols_ = list(set(low_var_cols + v_to_drop))
        
        # Save the "survivor" column names in order
        self.feature_cols_ = [c for c in df.columns if c not in self.drop_cols_]

        # MLflow Logging
        with mlflow.start_run(run_name='DecisionTree_Feature_Selection', nested=True):
            mlflow.log_param('variance_threshold', self.variance_threshold)
            mlflow.log_metric('v_nan_groups', len(null_patterns))
            mlflow.log_metric('total_features_dropped', len(self.drop_cols_))
            mlflow.log_metric('final_feature_count', len(self.feature_cols_))
        print(f"[FeatureSelector] Done — dropped {len(self.drop_cols_)} features, {len(self.feature_cols_)} remaining")

        return self

    def transform(self, X):
        print("select trans in")

        df = X.copy()
        
        # Drop the identified useless columns
        df.drop(columns=[c for c in self.drop_cols_ if c in df.columns], 
                inplace=True, errors='ignore')
        
        df = df.reindex(columns=self.feature_cols_, fill_value=0)
        
        return df

### Preprocessing Pipeline

In [68]:
from sklearn.pipeline import Pipeline

NULL_THRESH = 0.8
OHE_THRESH  = 10
VAR_THRESH  = 0.01


import mlflow
import mlflow.sklearn

mlflow.set_experiment("DecisionTree_Training")

preprocessing = Pipeline([
    ('cleaning',     Cleaner(null_threshold=NULL_THRESH)),
    ('feature_eng',  FeatureEngineer()),
    ('encoding',     CategoricalEncoder(ohe_threshold=OHE_THRESH)),
    ('selection',    FeatureSelector(variance_threshold=VAR_THRESH)),
])



In [69]:
X_train = train_df.drop(columns=['isFraud', 'TransactionID'])
y_train = train_df['isFraud']

X_val = val_df.drop(columns=['isFraud', 'TransactionID'])
y_val = val_df['isFraud']


In [70]:
X_train = preprocessing.fit_transform(X_train, y_train)
X_val   = preprocessing.transform(X_val)

print("Preprocessing Done")

clean_in
🏃 View run DecisionTree_Cleaning at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/60731f566555478d9d21eb4d9ca73377
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5
[Cleaner] Done — dropped 74 cols, 472432 rows
clean trans in
🏃 View run DecisionTree_Engineering at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/c094ccf31bb24fcd9ee92b835cad90e1
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5
encode_in
🏃 View run DecisionTree_Categorical_Encoding at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/e8427e4fe214460f9af1077a406a7945
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5
[CategoricalEncoder] Done — OHE cols: 12, LE cols: 4
encode trans in
select_in
🏃 View run DecisionTree_Feature_Selection at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/76914d4d6c014ece9b838a

## Training

In [46]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit


In [47]:
# Class imbalance weights
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
class_weight = {0: 1.0, 1: neg / pos}


In [48]:
with mlflow.start_run(run_name="DT_Underfitted"):

    params = {
        'max_depth': 2,
        'min_samples_leaf': 1,
        'class_weight': class_weight,
        'random_state': 42,
    }
    model = DecisionTreeClassifier(**params)
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])

    mlflow.log_params({k: str(v) for k, v in params.items()})
    mlflow.log_metric("train_auc",   train_auc)
    mlflow.log_metric("val_auc",     val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)

    print(f"UNDERFITTED — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"Gap: {train_auc - val_auc:.4f}")

UNDERFITTED — train: 0.7155, val: 0.7028
Gap: 0.0127
🏃 View run DT_Underfitted at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/cf42698589da4174bfc2be1db36860ab
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5


In [49]:
with mlflow.start_run(run_name="DT_Overfitted"):

    params = {
        'max_depth': None,      # unlimited depth — will memorize
        'min_samples_leaf': 1,
        'class_weight': class_weight,
        'random_state': 42,
    }
    model = DecisionTreeClassifier(**params)
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])

    mlflow.log_params({k: str(v) for k, v in params.items()})
    mlflow.log_metric("train_auc",   train_auc)
    mlflow.log_metric("val_auc",     val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)

    print(f"OVERFITTED — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"Gap: {train_auc - val_auc:.4f}")


OVERFITTED — train: 1.0000, val: 0.6076
Gap: 0.3924
🏃 View run DT_Overfitted at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/354f3bb923e54cbf98f398baad821ba0
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5


In [50]:
with mlflow.start_run(run_name="DT_Baseline") as run:

    params = {
        'max_depth': 10,
        'min_samples_leaf': 20,
        'class_weight': class_weight,
        'random_state': 42,
    }

    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []

    for fold, (tr_idx, cv_idx) in enumerate(tscv.split(X_train)):
        fold_model = DecisionTreeClassifier(**params)
        fold_model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        score = roc_auc_score(y_train.iloc[cv_idx],
                              fold_model.predict_proba(X_train.iloc[cv_idx])[:, 1])
        cv_scores.append(score)
        print(f"  Fold {fold+1}: {score:.4f}")

    model = DecisionTreeClassifier(**params)
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])

    mlflow.log_params({k: str(v) for k, v in params.items()})
    mlflow.log_metric("train_auc",   train_auc)
    mlflow.log_metric("val_auc",     val_auc)
    mlflow.log_metric("cv_mean_auc", np.mean(cv_scores))
    mlflow.log_metric("cv_std_auc",  np.std(cv_scores))
    mlflow.log_metric("overfit_gap", train_auc - val_auc)

    baseline_run_id = run.info.run_id

    print(f"\nBASELINE — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"CV       — mean: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}")
    print(f"Gap      — {train_auc - val_auc:.4f}")

  Fold 1: 0.7330
  Fold 2: 0.7407
  Fold 3: 0.7971
  Fold 4: 0.8044
  Fold 5: 0.8410

BASELINE — train: 0.9085, val: 0.8054
CV       — mean: 0.7832, std: 0.0408
Gap      — 0.1031
🏃 View run DT_Baseline at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/c3e9da7176734b55b58f7251dd5d9af9
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5


In [51]:
with mlflow.start_run(run_name="DT_Tuned") as run:

    params = {
        'max_depth': 8,
        'min_samples_leaf': 50,
        'min_samples_split': 100,
        'max_features': 'sqrt',
        'class_weight': class_weight,
        'random_state': 42,
    }

    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []

    for fold, (tr_idx, cv_idx) in enumerate(tscv.split(X_train)):
        fold_model = DecisionTreeClassifier(**params)
        fold_model.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        score = roc_auc_score(y_train.iloc[cv_idx],
                              fold_model.predict_proba(X_train.iloc[cv_idx])[:, 1])
        cv_scores.append(score)
        print(f"  Fold {fold+1}: {score:.4f}")

    model = DecisionTreeClassifier(**params)
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])

    mlflow.log_params({k: str(v) for k, v in params.items()})
    mlflow.log_metric("train_auc",   train_auc)
    mlflow.log_metric("val_auc",     val_auc)
    mlflow.log_metric("cv_mean_auc", np.mean(cv_scores))
    mlflow.log_metric("cv_std_auc",  np.std(cv_scores))
    mlflow.log_metric("overfit_gap", train_auc - val_auc)

    tuned_run_id = run.info.run_id

    print(f"\nTUNED — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"CV    — mean: {np.mean(cv_scores):.4f}, std: {np.std(cv_scores):.4f}")
    print(f"Gap   — {train_auc - val_auc:.4f}")

  Fold 1: 0.7702
  Fold 2: 0.7918
  Fold 3: 0.8199
  Fold 4: 0.8116
  Fold 5: 0.8355

TUNED — train: 0.8649, val: 0.8059
CV    — mean: 0.8058, std: 0.0227
Gap   — 0.0590
🏃 View run DT_Tuned at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/9ca00fcb8ffb44fc96ba7ba84c26cb96
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5


In [52]:
with mlflow.start_run(run_name="DT_NoClassWeight"):

    params = {
        'max_depth': 8,
        'min_samples_leaf': 50,
        'min_samples_split': 100,
        'max_features': 'sqrt',
        'random_state': 42,
    }
    model = DecisionTreeClassifier(**params)
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])

    mlflow.log_params({k: str(v) for k, v in params.items()})
    mlflow.log_param('class_weighting', False)
    mlflow.log_metric("train_auc",   train_auc)
    mlflow.log_metric("val_auc",     val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)

    print(f"NO CLASS WEIGHT — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print("Compare val_auc vs Tuned to see impact of class weighting")

NO CLASS WEIGHT — train: 0.8384, val: 0.8119
Compare val_auc vs Tuned to see impact of class weighting
🏃 View run DT_NoClassWeight at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/11ceb21d62264fe78911b83436f96633
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5


In [53]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
X_train_us, y_train_us = rus.fit_resample(X_train, y_train)

print(f"Before — fraud: {y_train.sum():,}, non-fraud: {(y_train==0).sum():,}")
print(f"After  — fraud: {y_train_us.sum():,}, non-fraud: {(y_train_us==0).sum():,}")
print(f"New shape: {X_train_us.shape}")

Before — fraud: 16,599, non-fraud: 455,833
After  — fraud: 16,599, non-fraud: 165,990
New shape: (182589, 180)


In [54]:
with mlflow.start_run(run_name="DT_undersampling"):

    params = {
        'max_depth': 12,
        'min_samples_leaf': 120,
        'min_samples_split': 60,
        'max_features': 'sqrt',
        'random_state': 42,
    }
    model = DecisionTreeClassifier(**params)
    model.fit(X_train_us, y_train_us)

    train_auc = roc_auc_score(y_train_us, model.predict_proba(X_train_us)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])

    mlflow.log_params({k: str(v) for k, v in params.items()})
    mlflow.log_param('class_weighting', False)
    mlflow.log_metric("train_auc",   train_auc)
    mlflow.log_metric("val_auc",     val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)

    print(f"NO CLASS WEIGHT — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print("Compare val_auc vs Tuned to see impact of class weighting")

NO CLASS WEIGHT — train: 0.8816, val: 0.8234
Compare val_auc vs Tuned to see impact of class weighting
🏃 View run DT_undersampling at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/0cd59125745041649e57f075795e07cd
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5


In [62]:
with mlflow.start_run(run_name="DT_balancing"):

    params = {
        'max_depth': 12,
        'min_samples_leaf': 120,
        'min_samples_split': 60,
        'max_features': 'sqrt',
        'random_state': 42,
        'class_weight': 'balanced',

    }
    model = DecisionTreeClassifier(**params)
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    val_auc   = roc_auc_score(y_val,   model.predict_proba(X_val)[:, 1])

    mlflow.log_params({k: str(v) for k, v in params.items()})
    mlflow.log_param('class_weighting', False)
    mlflow.log_metric("train_auc",   train_auc)
    mlflow.log_metric("val_auc",     val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)

    print(f"NO CLASS WEIGHT — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print("Compare val_auc vs Tuned to see impact of class weighting")

NO CLASS WEIGHT — train: 0.9044, val: 0.8321
Compare val_auc vs Tuned to see impact of class weighting
🏃 View run DT_balancing at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/160f81b0fe824ef9b744d4d7935bbd3e
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5


In [93]:
with mlflow.start_run(run_name="DT_Tuned_us") as run:

    params = {
        'max_depth': 8,
        'min_samples_leaf': 50,
        'min_samples_split': 80,
        'max_features': 0.5,
        'class_weight': class_weight,
        'random_state': 42,
    }

    model = DecisionTreeClassifier(**params)
    model.fit(X_train_us, y_train_us)

    X_val_aligned = X_val.reindex(columns=X_train_us.columns, fill_value=-999)

    train_auc = roc_auc_score(
        y_train_us,
        model.predict_proba(X_train_us)[:, 1]
    )

    val_auc = roc_auc_score(
        y_val,
        model.predict_proba(X_val_aligned)[:, 1]
    )

    mlflow.log_params({k: str(v) for k, v in params.items()})
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("val_auc", val_auc)
    mlflow.log_metric("overfit_gap", train_auc - val_auc)

    print(f"TUNED — train: {train_auc:.4f}, val: {val_auc:.4f}")
    print(f"Gap — {train_auc - val_auc:.4f}")

TUNED — train: 0.8810, val: 0.8345
Gap — 0.0465
🏃 View run DT_Tuned_us at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/16ed6d33067f4e9fa51bdb3a2179407e
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5


In [98]:
with mlflow.start_run(run_name='DT_FullPipeline_Registry') as run:
    print('Building full pipeline...')
    full_pipeline = Pipeline([
        ('cleaning',    preprocessing.named_steps['cleaning']),
        ('feature_eng', preprocessing.named_steps['feature_eng']),
        ('encoding',    preprocessing.named_steps['encoding']),
        ('selection',   preprocessing.named_steps['selection']),
        ('model',       final_model)
    ])

    print('Verifying pipeline on raw sample...')
    raw_sample = train_df.iloc[:100].drop(columns=['isFraud', 'TransactionID'])
    test_preds = full_pipeline.predict_proba(raw_sample)[:, 1]
    print(f'Sample preds: min={test_preds.min():.3f}, max={test_preds.max():.3f}')
    print('Pipeline verified!')

    best_params = {
        'max_depth': 8,
        'min_samples_leaf': 50,
        'min_samples_split': 80,
        'max_features': 0.5,
        'class_weight': class_weight,
        'random_state': 42,
    }
    mlflow.log_params({k: str(v) for k, v in best_params.items()})
    mlflow.log_param('pipeline_steps', 'Cleaner→FeatureEngineer→CategoricalEncoder→FeatureSelector→DT')
    mlflow.log_param('balancing', 'RandomUnderSampling_0.1')
    mlflow.log_metric('train_auc', roc_auc_score(y_us_final, final_model.predict_proba(X_us_final)[:, 1]))
    mlflow.log_metric('val_auc',   roc_auc_score(y_val, final_model.predict_proba(X_val)[:, 1]))

    mlflow.sklearn.log_model(
        sk_model=full_pipeline,
        name='full_dt_pipeline'
    )

    dt_pipeline_run_id = run.info.run_id
    print(f'\nFull pipeline saved!')
    print(f'Run ID: {dt_pipeline_run_id}')

Building full pipeline...
Verifying pipeline on raw sample...
clean trans in
encode trans in
select trans in
Sample preds: min=0.000, max=0.901
Pipeline verified!


2026/05/06 12:30:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Full pipeline saved!
Run ID: d4aa2a64e84940419ea5ff69e5a2a5ef
🏃 View run DT_FullPipeline_Registry at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5/runs/d4aa2a64e84940419ea5ff69e5a2a5ef
🧪 View experiment at: https://dagshub.com/gvakh23/ML_assignment2.mlflow/#/experiments/5
